<a href="https://colab.research.google.com/github/hy961/Case-Studies-for-Data-Science/blob/main/Case_Studies_for_Data_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dataset 2 HateXplain
# Models used SVM (LinearSVC) and  Logistic Regression (as baseline)

import json
import pandas as pd
from collections import Counter
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from google.colab import files

RANDOM_STATE = 999

# Upload the dataset
from google.colab import files
uploaded = files.upload()

# Import the dataset
with open("dataset.json", "r") as f:
    raw = json.load(f)

# Preprocessing
#    Each post has 3 annotators. Take the majority label/ Drop posts where all 3 disagree (no clear majority)
#    Then, reconstruct text from the token list.

rows = []
dropped = 0
for post_id, entry in raw.items():
    labels = [ann["label"] for ann in entry["annotators"]]
    majority_label, count = Counter(labels).most_common(1)[0]
    if count >= 2:
        rows.append({
            "post_id": post_id,
            "text": " ".join(entry["post_tokens"]),
            "label": majority_label
        })
    else:
        dropped += 1

hx = pd.DataFrame(rows)

print("Dataset Summary")
print("=" * 50)
print(f"Total posts in raw file : {len(raw)}")
print(f"Dropped (no majority)   : {dropped}")
print(f"Retained for modelling  : {len(hx)}")
print()
print("Class distribution:")
print(hx["label"].value_counts())
print()
print("Sample rows:")
print(hx.head())
print()

# Save cleaned version
hx.to_csv("hatexplain_clean.csv", index=False)


# Train/test split & TF-IDF
X_train, X_test, y_train, y_test = train_test_split(
    hx["text"], hx["label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=hx["label"]
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape :", X_test_tfidf.shape)
print()

# Model 1: SVM ( it is a new algorithm, not used in previous  ML course)
svm = LinearSVC(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=5000
)
svm.fit(X_train_tfidf, y_train)
svm_preds = svm.predict(X_test_tfidf)

print("=" * 50)
print("Model 1: SVM (LinearSVC)")
print("=" * 50)
print(classification_report(y_test, svm_preds, digits=3))
print("Macro F1:", round(f1_score(y_test, svm_preds, average="macro"), 3))
print()
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, svm_preds, labels=sorted(hx["label"].unique())),
    index=sorted(hx["label"].unique()),
    columns=sorted(hx["label"].unique())
))
print()

feats = np.array(vectorizer.get_feature_names_out())

print("=" * 50)
print("Top weighted features per class (SVM)")
print("=" * 50)
for i, cls in enumerate(svm.classes_):
    top = np.argsort(svm.coef_[i])[::-1][:12]
    print(f"\n{cls}:")
    print("  " + ", ".join(feats[top]))
print()

# Model 2: Logistic Regression
logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
logreg.fit(X_train_tfidf, y_train)
logreg_preds = logreg.predict(X_test_tfidf)

print("=" * 50)
print("Model 2: Logistic Regression")
print("=" * 50)
print(classification_report(y_test, logreg_preds, digits=3))
print("Macro F1:", round(f1_score(y_test, logreg_preds, average="macro"), 3))
print()
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, logreg_preds, labels=sorted(hx["label"].unique())),
    index=sorted(hx["label"].unique()),
    columns=sorted(hx["label"].unique())
))
print()


# Summaries
summary = pd.DataFrame({
    "Model": ["SVM (LinearSVC)", "Logistic Regression"],
    "Accuracy": [
        round((svm_preds == y_test).mean(), 3),
        round((logreg_preds == y_test).mean(), 3)
    ],
    "Macro F1": [
        round(f1_score(y_test, svm_preds, average="macro"), 3),
        round(f1_score(y_test, logreg_preds, average="macro"), 3)
    ],
    "Weighted F1": [
        round(f1_score(y_test, svm_preds, average="weighted"), 3),
        round(f1_score(y_test, logreg_preds, average="weighted"), 3)
    ]
})

print("=" * 50)
print("Summary of Dataset 2")
print("=" * 50)
print(summary.to_string(index=False))

Saving dataset.json to dataset.json
Dataset Summary
Total posts in raw file : 20148
Dropped (no majority)   : 919
Retained for modelling  : 19229

Class distribution:
label
normal        7814
hatespeech    5935
offensive     5480
Name: count, dtype: int64

Sample rows:
                       post_id  \
0  1179055004553900032_twitter   
1  1179063826874032128_twitter   
2  1178793830532956161_twitter   
3  1179088797964763136_twitter   
4  1179085312976445440_twitter   

                                                text       label  
0  i dont think im getting my baby them white 9 h...      normal  
1  we cannot continue calling ourselves feminists...      normal  
2                      nawt yall niggers ignoring me      normal  
3  <user> i am bit confused coz chinese ppl can n...  hatespeech  
4  this bitch in whataburger eating a burger with...  hatespeech  

Train shape: (15383, 10000)
Test shape : (3846, 10000)

Model 1: SVM (LinearSVC)
              precision    recall  f1-sco

In [ ]:
# Dataset 1 Jigsaw Toxic Comment Classification
# Models used SVM (LinearSVC) and Logistic Regression

import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

RANDOM_STATE = 999

from google.colab import files
uploaded = files.upload()

jig = pd.read_csv("train.csv")
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Dataset Summary")
print("=" * 50)
print("Shape:", jig.shape)
print("Columns:", list(jig.columns))
print("\nPositive count per label:")
print(jig[label_cols].sum())

n_clean = (jig[label_cols].sum(axis=1) == 0).sum()
print(f"\nComments with NO label: {n_clean} ({n_clean / len(jig):.1%})")
print("\nLabels per comment (how many carry multiple):")
print(jig[label_cols].sum(axis=1).value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    jig["comment_text"], jig[label_cols],
    test_size=0.2,
    random_state=RANDOM_STATE
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

# Using OneVsRestClassifier here as this is multi-label, a comment ()
# It can be toxic & obscene & an insult all at the same time
results = []

def run_model(name, base_estimator):
    clf = OneVsRestClassifier(base_estimator)
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)

    print(f"\n{name} , Dataset 1")
    print(classification_report(
        y_test, preds, target_names=label_cols, digits=3, zero_division=0
    ))

    micro_f1 = f1_score(y_test, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    micro_p = precision_score(y_test, preds, average="micro", zero_division=0)
    micro_r = recall_score(y_test, preds, average="micro", zero_division=0)

    print("Micro Precision:", round(micro_p, 3))
    print("Micro Recall:", round(micro_r, 3))
    print("Micro F1:", round(micro_f1, 3))
    print("Macro F1:", round(macro_f1, 3))

    results.append({
        "Model": name,
        "Precision": round(micro_p, 3),
        "Recall": round(micro_r, 3),
        "Micro F1": round(micro_f1, 3),
        "Macro F1": round(macro_f1, 3)
    })

    return clf

# Using balanced class weights here, compensates for the heavy imbalance
# and pushes recall up
run_model(
    "SVM (LinearSVC, balanced)",
    LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
)

run_model(
    "Logistic Regression (balanced)",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
)

# No class balancing this round. expecting precision to rise and
# recall to fall, which is the trade-off will be covered in the report
svm_default = run_model(
    "SVM (LinearSVC, default)",
    LinearSVC(max_iter=5000, random_state=RANDOM_STATE)
)

run_model(
    "Logistic Regression (default)",
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)

feats = np.array(vectorizer.get_feature_names_out())

print("=" * 50)
print("Top weighted features per label (SVM, default weights)")
print("=" * 50)
for i, lab in enumerate(label_cols):
    coefs = svm_default.estimators_[i].coef_[0]
    top = np.argsort(coefs)[::-1][:12]
    print(f"\n{lab}:")
    print("  " + "0, ".join(feats[top]))
print()

summary = pd.DataFrame(results)

print("=" * 50)
print("\nSummary of Dataset 1")
print("=" * 50)
print(summary.to_string(index=False))

Saving train.csv to train.csv
Dataset Summary
Shape: (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Positive count per label:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Comments with NO label: 143346 (89.8%)

Labels per comment (how many carry multiple):
0    143346
1      6360
2      3480
3      4209
4      1760
5       385
6        31
Name: count, dtype: int64
Train shape: (127656, 10000)
Test shape: (31915, 10000)

SVM (LinearSVC, balanced) , Dataset 1
               precision    recall  f1-score   support

        toxic      0.585     0.839     0.689      3169
 severe_toxic      0.251     0.696     0.369       326
      obscene      0.599     0.852     0.704      1719
       threat      0.304     0.546     0.391       108
       insult      0.474     0.828     0.603      1612
identity_hate      0.216     

In [ ]:

# setup, and rebuild the data with target-community annotations
!pip install fairlearn -q

import json, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 999

def vect():
    return TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                           stop_words="english")

def build_svm():
    return LinearSVC(class_weight="balanced", random_state=RANDOM_STATE,
                     max_iter=5000)

def build_logreg():
    return LogisticRegression(max_iter=2000, class_weight="balanced",
                              random_state=RANDOM_STATE)

# Same majority vote as Task 1, but also keep the target community that
# at least 2 of the 3 annotators agreed the post is about.
rows = []
for post_id, entry in raw.items():
    labels = [a["label"] for a in entry["annotators"]]
    lab, cnt = Counter(labels).most_common(1)[0]
    if cnt >= 2:
        tc = Counter()
        for a in entry["annotators"]:
            for t in set(a.get("target", [])):
                tc[t] += 1
        targets = sorted([t for t, c in tc.items()
                          if c >= 2 and t not in ("None", "Other")])
        rows.append({"post_id": post_id,
                     "text": " ".join(entry["post_tokens"]),
                     "label": lab,
                     "targets": targets})

hx2 = pd.DataFrame(rows)
print("Retained:", len(hx2))
print("Posts with a majority-agreed 'Indigenous' target:",
      hx2["targets"].apply(lambda ts: "Indigenous" in ts).sum())

# Experiment 1: 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, build in [("SVM (LinearSVC)", build_svm),
                    ("Logistic Regression", build_logreg)]:
    macro, per_class = [], []
    for tr, te in skf.split(hx2["text"], hx2["label"]):
        v = vect()
        Xtr = v.fit_transform(hx2["text"].iloc[tr])     # fit inside the fold
        Xte = v.transform(hx2["text"].iloc[te])         # no leakage
        m = build()
        m.fit(Xtr, hx2["label"].iloc[tr])
        p = m.predict(Xte)
        macro.append(f1_score(hx2["label"].iloc[te], p, average="macro"))
        per_class.append(f1_score(hx2["label"].iloc[te], p, average=None,
                                  labels=["hatespeech", "normal", "offensive"]))
    macro, per_class = np.array(macro), np.array(per_class)
    print(f"\n{name}")
    print("  per fold :", np.round(macro, 3).tolist())
    print(f"  macro F1 : {macro.mean():.3f} +/- {macro.std():.3f}")
    for i, c in enumerate(["hatespeech", "normal", "offensive"]):
        print(f"    {c:11s} {per_class[:,i].mean():.3f} +/- {per_class[:,i].std():.3f}")

# Experiment 2: learning curves

Xtr_pool, X_test2, ytr_pool, y_test2 = train_test_split(
    hx2["text"], hx2["label"], test_size=0.2,
    random_state=RANDOM_STATE, stratify=hx2["label"])
Xtr_pool = Xtr_pool.reset_index(drop=True)
ytr_pool = ytr_pool.reset_index(drop=True)

sizes = [500, 1000, 2000, 4000, 8000, 12000, len(Xtr_pool)]
curves = {}

for name, build in [("SVM (LinearSVC)", build_svm),
                    ("Logistic Regression", build_logreg)]:
    means, cls_means = [], []
    print(f"\n{name}")
    for n in sizes:
        scores, cls = [], []
        reps = 1 if n == len(Xtr_pool) else 3     # 3 subsamples per size
        for r in range(reps):
            idx = np.random.RandomState(RANDOM_STATE + r).choice(
                len(Xtr_pool), n, replace=False)
            v = vect()
            Xtr = v.fit_transform(Xtr_pool.iloc[idx])
            Xte = v.transform(X_test2)
            m = build()
            m.fit(Xtr, ytr_pool.iloc[idx])
            p = m.predict(Xte)
            scores.append(f1_score(y_test2, p, average="macro"))
            cls.append(f1_score(y_test2, p, average=None,
                                labels=["hatespeech", "normal", "offensive"]))
        means.append(np.mean(scores))
        cls_means.append(np.mean(cls, axis=0))
        print(f"  n={n:6d}  macro F1 = {np.mean(scores):.3f}"
              f"  (hate {np.mean(cls,axis=0)[0]:.3f} |"
              f" offensive {np.mean(cls,axis=0)[2]:.3f})")
    curves[name] = (means, np.array(cls_means))

# the figure used in the report
svm_macro = curves["SVM (LinearSVC)"][0]
lr_macro, lr_cls = curves["Logistic Regression"][0], curves["Logistic Regression"][1]

fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.plot(sizes, lr_macro, marker="o", color="#1f3b73",
        label="Logistic regression, macro F1")
ax.plot(sizes, svm_macro, marker="o", color="#7a90b8", label="SVM, macro F1")
ax.plot(sizes, lr_cls[:, 0], marker="s", ms=4, ls="--", color="#b4413c",
        label="LR, hate speech class")
ax.plot(sizes, lr_cls[:, 2], marker="v", ms=4, ls="--", color="#c07a20",
        label="LR, offensive class")
ax.set_xlabel("Training examples"); ax.set_ylabel("F1")
ax.grid(alpha=.3); ax.set_ylim(0.35, 0.78)
ax.legend(fontsize=7.5, loc="lower right")
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=170, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("learning_curves.png")     # upload this to Overleaf



# Experiment 3: fairness audit by target community (Fairlearn)

from fairlearn.metrics import (MetricFrame, false_positive_rate,
                               false_negative_rate, selection_rate)

v = vect()
Xtr = v.fit_transform(Xtr_pool)
Xte = v.transform(X_test2)
model = build_logreg()
model.fit(Xtr, ytr_pool)
pred = model.predict(Xte)

# operational decision: flag (hatespeech/offensive) vs leave up (normal)
y_true_bin = (y_test2 != "normal").astype(int).values
y_pred_bin = (pred != "normal").astype(int)
test_targets = hx2.loc[X_test2.index, "targets"]

def boot_ci(y_true, y_pred, fn, n=2000, seed=1):
    rng = np.random.RandomState(seed); out = []
    for _ in range(n):
        i = rng.randint(0, len(y_true), len(y_true))
        try: out.append(fn(y_true[i], y_pred[i]))
        except Exception: pass
    return (np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5))

GROUPS = ["African", "Islam", "Jewish", "Homosexual", "Women", "Refugee",
          "Arab", "Caucasian", "Asian", "Hispanic"]

print(f"{'group':15s} {'n':>5s} {'n_nontox':>9s} {'FPR':>6s} {'95% CI':>16s}"
      f" {'FNR':>6s} {'95% CI':>16s}")
recs = []
for g in GROUPS + ["__none__"]:
    mask = (test_targets.apply(lambda ts: len(ts) == 0) if g == "__none__"
            else test_targets.apply(lambda ts: g in ts)).values
    a, b = y_true_bin[mask], y_pred_bin[mask]
    n_nt, n_t = int((a == 0).sum()), int((a == 1).sum())
    fpr = false_positive_rate(a, b) if n_nt else np.nan
    fnr = false_negative_rate(a, b) if n_t else np.nan
    lo1, hi1 = boot_ci(a, b, false_positive_rate) if n_nt >= 10 else (np.nan, np.nan)
    lo2, hi2 = boot_ci(a, b, false_negative_rate) if n_t >= 10 else (np.nan, np.nan)
    label = "no target named" if g == "__none__" else g
    print(f"{label:15s} {int(mask.sum()):5d} {n_nt:9d} {fpr:6.3f}"
          f" [{lo1:5.3f},{hi1:5.3f}] {fnr:6.3f} [{lo2:5.3f},{hi2:5.3f}]")
    recs.append(dict(group=label, n=int(mask.sum()), n_nontoxic=n_nt,
                     fpr=fpr, fpr_lo=lo1, fpr_hi=hi1,
                     n_toxic=n_t, fnr=fnr, fnr_lo=lo2, fnr_hi=hi2))

df_fair = pd.DataFrame(recs)
df_fair.to_csv("fairness_by_group.csv", index=False)

# Fairlearn disparity summary over the named communities
pairs = []
for g in GROUPS:
    mask = test_targets.apply(lambda ts: g in ts).values
    pairs += [(g, x, y) for x, y in zip(y_true_bin[mask], y_pred_bin[mask])]
fl = pd.DataFrame(pairs, columns=["g", "yt", "yp"])
mf = MetricFrame(metrics={"selection_rate": selection_rate,
                          "FPR": false_positive_rate,
                          "FNR": false_negative_rate},
                 y_true=fl.yt.values, y_pred=fl.yp.values,
                 sensitive_features=fl.g.values)
print("\nFairlearn MetricFrame by target community:")
print(mf.by_group.round(3).to_string())
print("\ndifference (max-min):", {k: round(v, 3) for k, v in mf.difference().items()})
print("ratio      (min/max):", {k: round(v, 3) for k, v in mf.ratio().items()})